### Redshift에 있는 aa_example 테이블 사용
- 실제 구인 사이트에서 가져온 익명화된 로그 데이터
- 일별 구직자 + JD Funnel data
``` sql
CREATE TABLE raw_data.aa_example (
user_id int,
date date,
job_position_id int,
clicked int,
checkedout int,
applied int
);
```

In [11]:
%pip install psycopg2-binary


Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [2]:
import psycopg2
import pandas as pd
import pandas.io.sql as sqlio
import math

# Redshift connection 함수
def get_Redshift_connection(autocommit):
    host = "learnde.cduaw970ssvt.ap-northeast-2.redshift.amazonaws.com"
    redshift_user = "jico9954"
    redshift_pass = "Jico9954!1"
    port = 5439
    dbname = "dev"
    conn = psycopg2.connect("dbname={dbname} user={user} host={host} password={password} port={port}".format(
        dbname=dbname,
        user=redshift_user,
        password=redshift_pass,
        host=host,
        port=port
    ))
    conn.set_session(autocommit=autocommit)
    return conn

In [5]:
conn = get_Redshift_connection(True)
cur = conn.cursor()
sql = '''SELECT * FROM raw_data.aa_example'''
df = sqlio.read_sql_query(sql, conn)

/var/folders/yr/8q3yc9wn6ln3ypw1ry6kgf280000gn/T/ipykernel_27985/1453264833.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(sql, conn)


In [6]:
df.head()

,user_id,date,job_position_id,clicked,checkedout,applied
0,6810,2022-01-03,4891,1,1,0
1,117686,2022-01-03,8882,1,1,1
2,117686,2022-01-03,9006,1,1,1
3,316741,2022-01-03,8331,1,1,1
4,380002,2022-01-03,2385,1,1,1


## Python

In [27]:
import hashlib # 문자열을 해시값으로 바꾸는 표준 라이브러리 

# 사용자 ID 값을 받아서 버킷 번호를 반환하는 함수
def split_userid(id): 
     """Given an id and the number of variants, returns a bucket number"""
     h = hashlib.md5(str(id).encode())
     # id를 문자열로 바꾸고 .encode()로 바이트로 바꾼 뒤 md5 해시를 계산

     return int(h.hexdigest(), 16) % 2 # num_of_variants
     # h.hexdigest()는 해시 결과를 16진수 문자열로 반환
     # int(..., 16)은 그 16진수 문자열을 정수로 변환
     # % 2는 2로 나눈 나머지를 구함

In [28]:
print(split_userid(100))
print(split_userid(101))

1
0


## 사용자 버킷팅 코드의 한계와 개선 방향

처음 본 버킷팅 코드는 개념 이해에는 좋지만, 실무에서 바로 쓰기에는 한계가 있다.

### 기존 코드

```python
def split_userid(id, num_of_variants=2):
    h = hashlib.md5(str(id).encode())
    return int(h.hexdigest(), 16) % num_of_variants
```

### 이 코드의 장점

- 같은 사용자에게 항상 같은 버킷을 준다
- 구현이 간단하다
- 50:50 분배를 빠르게 확인하기 좋다

### 이 코드의 한계

- 모든 사용자를 항상 실험에 태운다
  - 실무에서는 보통 1% -> 5% -> 10%처럼 점진적으로 커버리지를 늘린다
- 어떤 A/B 테스트를 하든 같은 사용자는 항상 같은 쪽으로 치우칠 수 있다
  - 예를 들어 어떤 사용자는 여러 실험에서 계속 A만 보거나 B만 볼 수 있다
- 실험별 독립성이 약하다
  - `user_id`만 쓰면 서로 다른 실험이 같은 버킷팅 결과를 공유하게 된다

### 왜 취약한가

실무에서는 사용자 버킷팅이 단순히 `A냐 B냐`를 정하는 문제가 아니다.

- 먼저 이 사용자를 이번 실험에 포함할지 정해야 하고
- 포함한다면 control/test 중 어디에 넣을지 정해야 하며
- 다른 실험과도 가능한 한 독립적으로 동작해야 한다

기존 코드는 `항상 50:50`만 지원하고, `실험 ID`가 없어서 서로 다른 실험이 같은 배정 패턴을 공유할 수 있다는 점이 약하다.

### 어떻게 개선할 수 있나

- `size_of_test`를 넣어 실험 커버리지를 조절한다
- `abtest_id`를 해시에 같이 넣어 실험마다 다른 배정 패턴을 만든다
- 실험에 포함되지 않으면 `-1`을 반환해 분석에서 쉽게 제외할 수 있게 한다

즉, 실무형 버킷팅은 보통 아래 두 단계를 분리해서 생각한다.

1. 이 사용자를 이번 실험에 포함할 것인가
2. 포함된다면 어떤 variant에 넣을 것인가


In [ ]:
# 실무형에 더 가까운 버킷팅 예시
def split_userid_v2(abtest_id, user_id, size_of_test, num_of_variants=2):
    """실험 포함 여부와 variant 배정을 함께 처리한다."""
    mixed_id = f"{abtest_id}_{user_id}"
    h = hashlib.md5(mixed_id.encode())
    hashed_value = int(h.hexdigest(), 16)

    # 0~99 구간 중 일부만 이번 테스트에 포함
    if (hashed_value % 100) >= size_of_test:
        return -1

    # 테스트 대상에 포함되면 variant 배정
    return hashed_value % num_of_variants

print(split_userid_v2("checkout_v1", 100, 5))
print(split_userid_v2("checkout_v1", 101, 5))
print(split_userid_v2("recommendation_v2", 100, 5))


### 왜 `abtest_id`를 같이 섞어야 하나

`user_id`만 해시하면 같은 사용자는 여러 실험에서 비슷한 버킷 패턴을 반복해서 가질 수 있다.
그러면 어떤 사용자는 여러 실험에서 계속 A 쪽으로, 또 어떤 사용자는 계속 B 쪽으로 들어갈 가능성이 생긴다.

반면 `abtest_id`를 같이 섞으면 실험마다 다른 해시 입력이 만들어진다.
즉, 같은 사용자라도 실험 1과 실험 2에서 서로 다른 버킷 결과를 받을 수 있어서 실험 간 독립성이 더 좋아진다.

핵심은 `사용자 고정성`은 유지하면서도 `실험별로는 다른 배정 패턴`을 만드는 것이다.


In [ ]:
def split_userid_basic(user_id, num_of_variants=2):
    h = hashlib.md5(str(user_id).encode())
    return int(h.hexdigest(), 16) % num_of_variants

sample_user_id = 100

print("user_id만 사용:")
print("checkout_v1 ->", split_userid_basic(sample_user_id))
print("recommendation_v2 ->", split_userid_basic(sample_user_id))

print("\nabtest_id를 함께 사용:")
print("checkout_v1 ->", split_userid_v2("checkout_v1", sample_user_id, 100))
print("recommendation_v2 ->", split_userid_v2("recommendation_v2", sample_user_id, 100))


## SQL

In [23]:
sql = """
SELECT 100 user_id, MOD(STRTOL(LEFT(MD5(100),15), 16), 2) variant_id
UNION
SELECT 101 user_id,MOD(STRTOL(LEFT(MD5(101),15), 16), 2) variant_id
"""
df = sqlio.read_sql_query(sql, conn)

/var/folders/yr/8q3yc9wn6ln3ypw1ry6kgf280000gn/T/ipykernel_27985/724378767.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(sql, conn)


In [24]:
df.head()

,user_id,variant_id
0,100,1
1,101,0


## A/A Test 비교: SQL

- 아래 SQL을 수정해서 A와 B에 속한 사용자의 수의 카운트해 보세요

In [29]:
sql = """
SELECT 
    MOD(STRTOL(LEFT(MD5(user_id),15),16),2) variant_id,
    COUNT(DISTINCT user_id) user_sum,
    COUNT(1) session_sum
FROM (
    SELECT DISTINCT user_id, date
    FROM raw_data.aa_example
)
GROUP BY 1;
"""
df = sqlio.read_sql_query(sql, conn)

/var/folders/yr/8q3yc9wn6ln3ypw1ry6kgf280000gn/T/ipykernel_27985/1477341032.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(sql, conn)


In [15]:
df.head(5)

,variant_id,user_sum,session_sum
0,1,2056,2950
1,0,2070,3019


## A/A Test 비교: Python

- 아래 for 루프를 수정하여 (앞서 만든 split_userid 함수 호출) A와 B에 속한 사용자의 수를 카운트해 보세요

In [32]:
sql = """ SELECT DISTINCT user_id
FROM raw_data.aa_example
"""
user_df = sqlio.read_sql_query(sql, conn)

/var/folders/yr/8q3yc9wn6ln3ypw1ry6kgf280000gn/T/ipykernel_27985/390973837.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  user_df = sqlio.read_sql_query(sql, conn)


In [33]:
user_df.head()

,user_id
0,6810
1,117686
2,316741
3,380002
4,374410


In [34]:
import hashlib

def split_userid(id, num_of_variants=2):
   h = hashlib.md5(str(id).encode())
   return int(h.hexdigest(), 16) % num_of_variants

In [35]:
a_user_count = 0
b_user_count = 0

for index, row in user_df.iterrows():
    if split_userid(row["user_id"]) == 0:
        a_user_count += 1
    else:
        b_user_count += 1

print(a_user_count, b_user_count)

2104 2022


In [36]:
sql = """SELECT DISTINCT user_id, date
FROM raw_data.aa_example
"""
df = sqlio.read_sql_query(sql, conn)

/var/folders/yr/8q3yc9wn6ln3ypw1ry6kgf280000gn/T/ipykernel_27985/2737284253.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(sql, conn)


In [37]:
a_session_count = 0
b_session_count = 0

for index, row in df.iterrows():
    if split_userid(row["user_id"]) == 0:
        a_session_count += 1
    else:
        b_session_count += 1

print(a_session_count, b_session_count)

3027 2942


### 조금더 완전한 버킷팅 함수 - Production쪽에서 runtime에서 특정 사용자/브라우저를 A와 B중 어디에 넣을지 결정하는데 사용

In [18]:
def split_userid(id, num_of_variants=2, b_percent=50):
     """Given an id and the number of variants, returns a bucket number"""
     h = hashlib.md5(str(id).encode())
     val = int(h.hexdigest(), 16) % 100 # um_of_variants
     if val < b_percent:
       return 0
     else:
       return 1

In [19]:
sql = """
SELECT variant_id, COUNT(DISTINCT user_id) user_count, COUNT(1) session_count
FROM (
  SELECT
    user_id,
    MOD(STRTOL(LEFT(MD5(user_id),15), 16), 2) variant_id
  FROM raw_data.aa_example
)
GROUP BY 1
"""
df = sqlio.read_sql_query(sql, conn)

/var/folders/yr/8q3yc9wn6ln3ypw1ry6kgf280000gn/T/ipykernel_27985/3971874306.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(sql, conn)


In [20]:
df.head()

,variant_id,user_count,session_count
0,1,2056,10231
1,0,2070,10541
